# DINOv3 

This notebook used DINOv3 in order to retrieve embeddings for sections of the image.  The first code cell tests the model to extract the 
embedding and is basically slightly modified code from the website tutorial on hugging face.

In [1]:
import matplotlib.pyplot as plt
from kitty import depth_read, listPicsWith, togglePath, MatchDepthToCar, KITTY_PATH
from yolo import getEmbedFromResults, getCropsFromResults, getEmbedFromCrops, CLASSES_YOLO, CONFIDENCE_YOLO
from ultralytics import YOLO
from PIL import Image
import numpy as np
import time
import torch
from utils import getDevice

YOLO26n MODEL LOADED

Ultralytics Solutions: ✅ {'source': None, 'model': '../build/yolo26n.pt', 'classes': [2], 'show_conf': True, 'show_labels': True, 'region': None, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': '../imgs/crops/', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': False, 'iou': 0.7, 'conf': 0.5, 'device': None, 'max_det': 300, 'half': False, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/tim/sidehustle/thesis/.venv/lib64/python3.14/site-packages/cv2/qt/plugins"
QFont::fromString: Invalid description 'JetBrainsMonoNL Nerd Font,10,-1,5,400,0,0,0,0,0,0,0,0,0,0,1'
QFont::fromString: Invalid description 'JetBrainsMonoNL Nerd Font,10,-1,5,400,0,0,0,0,0,0,0,0,0,0,1'
QFont::fromString: Invalid description 'JetBrainsMonoNL Nerd Font,10,-1,5,400,0,0,0,0,0,0,0,0,0,0,1'
QFont::fromString: Invalid description 'JetBrainsMonoNL Nerd Font,9,-1,5,400,0,0,0,0,0,0,0,0,0,0,1'
QFontDatabase: Cannot find font directory /home/tim/sidehustle/thesis/.venv/lib64/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tim/sidehustle/thesis/.venv/lib64/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.i

CROPPER LOADED

Using device: xpu :3


In [2]:

from transformers import AutoImageProcessor, AutoModel
from transformers.image_utils import load_image

#load a single image
image = load_image("../imgs/image1.jpg")
device = getDevice()

DINO_MODEL_ID = "facebook/dinov3-vits16-pretrain-lvd1689m"

image_processor = AutoImageProcessor.from_pretrained(DINO_MODEL_ID)
dinov3 = AutoModel.from_pretrained(
    DINO_MODEL_ID,
    dtype=torch.float16,
    device_map="auto"
).to(device)

inputs = image_processor(images=image, return_tensors="pt").to(device)

dinov3.eval()
with torch.inference_mode():
    outputs = dinov3(**inputs)

#embeddings from output
global_embedding = outputs.pooler_output  # shape: [1, embed_dim]
print("Global embedding shape:", global_embedding.shape)

/home/tim/sidehustle/thesis/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: xpu :3


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 1991.23it/s]


Global embedding shape: torch.Size([1, 384])


In [3]:
def get_embedding(image):
    """
    Get DINOv3 embeddings from an image or a list of images. 
    Args:
        image: string path or PIL image or list of ready to pricess opened imaged
    """
    #case 1 - image is a path
    if type(image) == str:
        image = load_image(image)
    
    #case 2 is pil image already or compatible with the class
    inputs = image_processor(images=image, return_tensors="pt")

    #retrieve embeddings
    dinov3.eval()
    with torch.inference_mode():
        outputs = dinov3(**inputs)

    return outputs.pooler_output  # final embeddings

In [4]:
images_with_cars = listPicsWith(KITTY_PATH, CLASSES_YOLO, CONFIDENCE_YOLO, True)

model_yolo = YOLO("../build/yolo26n.pt")

results = model_yolo.predict(images_with_cars, conf=CONFIDENCE_YOLO, classes=CLASSES_YOLO)

crops = getCropsFromResults(results)

1000
2011_09_26_drive_0002_sync_image_0000000005_image_02.png
../datasets/depth_selection/val_selection_cropped/image/2011_09_26_drive_0002_sync_image_0000000005_image_02.png

0: 192x640 (no detections), 20.0ms
1: 192x640 (no detections), 20.0ms
2: 192x640 (no detections), 20.0ms
3: 192x640 (no detections), 20.0ms
4: 192x640 (no detections), 20.0ms
5: 192x640 (no detections), 20.0ms
6: 192x640 (no detections), 20.0ms
7: 192x640 (no detections), 20.0ms
8: 192x640 (no detections), 20.0ms
9: 192x640 (no detections), 20.0ms
10: 192x640 (no detections), 20.0ms
11: 192x640 (no detections), 20.0ms
12: 192x640 (no detections), 20.0ms
13: 192x640 (no detections), 20.0ms
14: 192x640 (no detections), 20.0ms
15: 192x640 (no detections), 20.0ms
16: 192x640 (no detections), 20.0ms
17: 192x640 (no detections), 20.0ms
18: 192x640 (no detections), 20.0ms
19: 192x640 (no detections), 20.0ms
20: 192x640 (no detections), 20.0ms
21: 192x640 1 car, 20.0ms
22: 192x640 1 car, 20.0ms
23: 192x640 1 car, 20.0ms


In [7]:
inputs = image_processor(images=crops, return_tensors="pt").to(device)

dinov3.eval()
with torch.inference_mode():
    outputs = dinov3(**inputs)

In [8]:
print("Crops embedding shape:", outputs.pooler_output.shape)

Crops embedding shape: torch.Size([2566, 384])
